# Lifting Equipment in Resident-Handling Injury Narratives

How often do employer-written narratives of resident-handling injuries in US nursing facilities mention patient-lifting equipment, and when they do, what part of the equipment's use is described?

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC)
**Data** OSHA ITA Case Detail Data, 2025 (submissions through 2026-08-31), NAICS 6231
**Source data** US Government work, 17 U.S.C. §105

### Disclosure

The author distributes a powered patient transfer system. No product, manufacturer, or brand is named, evaluated, or compared here. Equipment is grouped by type only. The question is descriptive: what the narratives say.

### What this analysis can and cannot show

- **Mention is not use.** A narrative that names a lift shows the lift was involved. A narrative that names no equipment does not show the transfer was manual; employers may simply not have written it down. Shares below are shares of narratives that mention equipment, not shares of transfers done with equipment.
- **No rates of equipment failure.** The data counts injuries, not transfers. It cannot say how often equipment is used safely, and it cannot compare equipment with manual handling.
- **No product judgment.** Injuries during lift use can come from the device, the setting, the task, the resident, or training. The narratives rarely distinguish these.

## 1. Environment and data

In [1]:
!pip install -q duckdb pandas requests
import os, re, duckdb, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)

CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_08-31-2026.csv'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
PATH = 'data/osha_cases.csv'
if not os.path.exists(PATH):
    r = requests.get(CASES_URL, headers=HDRS, timeout=1800, stream=True); r.raise_for_status()
    with open(PATH, 'wb') as f:
        for ch in r.iter_content(1 << 20): f.write(ch)

con = duckdb.connect(':memory:')
con.execute(f"CREATE TABLE raw AS SELECT * FROM read_csv_auto('{PATH}', all_varchar=true, ignore_errors=true, sample_size=-1)")
cols = [r[0] for r in con.execute('DESCRIBE raw').fetchall()]
low = {c.lower(): c for c in cols}
def col(*c): return next(low[x.lower()] for x in c if x.lower() in low)
nar = "lower(concat_ws(' ', " + ', '.join(f'"{col(c)}"' for c in
      ['new_nar_before_incident', 'new_nar_what_happened', 'new_nar_injury_illness', 'new_nar_object_substance']) + '))'
nf = con.execute(f'''SELECT {nar} AS narr,
       CASE WHEN TRY_CAST("{col('dafw_num_away')}" AS INT) >= 0 THEN TRY_CAST("{col('dafw_num_away')}" AS INT) END AS dafw
     FROM raw WHERE "{col('naics_code')}" LIKE '6231%' ''').df()
print(f'{len(nf):,} nursing-facility cases')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

25,860 nursing-facility cases


## 2. The frozen handling rule

Copied unchanged from [`../nursing-facility-handling/`](../nursing-facility-handling/).

In [2]:
PERSON   = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE   = r"patient handling|resident handling|resident care transfer"
n = nf.narr.fillna('')
nf['handling'] = (n.str.contains(PERSON, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
print(f"handling cases: {int(nf.handling.sum()):,} (should equal the nursing-facility-handling module: 5,654)")

handling cases: 5,654 (should equal the nursing-facility-handling module: 5,654)


## 3. Equipment mentioned in the narrative

Three groups, by type only:

- **Patient lift**: mechanical or powered lifts, including full-body and sit-to-stand lifts, and their slings and pads.
- **Low-tech aid**: gait belts, draw or slide sheets, transfer boards, and pads used to move a resident.
- **Vehicle lift**: wheelchair lifts on vans or buses. These are not patient-handling equipment and are counted separately.

The patterns match equipment words, not the verb "lift". Spot checks follow, and a blind validation sample is written in Section 6.

In [3]:
EQ_LIFT = (r"\b(?:hoyer|mechanical lift|(?:total|full[- ]body|ceiling|stand(?:ing)?[- ]?up|sit[- ]to[- ]stand)\s*lift|"
           r"sit[- ]to[- ]stand|stand[- ]?aid|stand[- ]?assist(?: device)?|lift sling|lift pad|sling|"
           r"(?:the|a|resident|patient|electric|battery) lift\b(?! (?:with|the resident|resident|patient|him|her))|"
           r"\] lift|lift (?:was|tipped|rolled|bumped|battery|malfunction\w*|failed|broke))")
EQ_AID  = r"\b(?:gait belt|draw ?sheet|slide ?sheet|slide ?board|transfer board|blue pad|bed pad|turn(?:ing)? sheet|lift pad)\b"
EQ_VEH  = r"\b(?:van|bus|vehicle|transport(?:ation)?)\b.{0,60}\blift\b|\blift\b.{0,60}\b(?:van|bus|vehicle)\b|wheelchair lift"
veh  = n.str.contains(EQ_VEH, regex=True)
lift = n.str.contains(EQ_LIFT, regex=True) & ~veh
aid  = n.str.contains(EQ_AID, regex=True) & ~lift
nf['equipment'] = np.select([lift, aid, veh], ['patient lift', 'low-tech aid', 'vehicle lift'], 'none mentioned')

tab = (nf.groupby(['handling', 'equipment']).size().unstack(0).fillna(0).astype(int)
         .rename(columns={True: 'handling cases', False: 'other cases'}))
tab['share of handling cases'] = (tab['handling cases'] / tab['handling cases'].sum()).round(3)
tab

handling,other cases,handling cases,share of handling cases
equipment,,,
low-tech aid,12,212,0.037
none mentioned,20083,4809,0.851
patient lift,109,615,0.109
vehicle lift,2,18,0.003


### Spot check: patient lift mentioned

In [4]:
for t in nf[nf.equipment == 'patient lift'].narr.sample(25, random_state=1): print(' -', t[:220])

 - pushing resident in [redacted] lift employee felt pain when pushing and pulling resident in [redacted] lift back lower bilateral musculoskeletal sprainstrain hoyer lift resident
 - administering medication [redacted] was assisting [redacted] in transfer of resident from bed to hoyer lift. [redacted] went to grab the hoyer left and felt a strain in her right hip strain or tear hoyer lift
 - employee was assisting resident in a lift. employee pincher finger in hoyer lift. contusion to the left ring finger hoyer lift
 - employee was assisting a resident with a sit to stand device when employee was bent to assist the resident felt a pull in their back back strain none- movement
 - she was getting a lift ready to transfer a resident she was bent over and when she stood up she hit her head on the bat above her concussion to the head the [redacted] bar on the lift
 - moving mechanical lift chair into patient room twisted ankle while moving mechanical lift twisted ankle moving mechanical ch

## 4. What part of lift use is described

Among narratives that mention a patient lift. Keyword groups; a case can fall in more than one. These describe what the employer wrote, not a cause established by investigation.

**Not validated; not cited.** The keyword groups in Section 4 were not checked against a review. Reading the spot check shows at least one known problem: the "catching or holding" group also matches staff who slipped or tripped near a lift, not only residents who fell. These counts are kept here for transparency and are not used in the README. The equipment groups in Section 3 were validated (see `equipment-review.ipynb`).

In [5]:
L = nf[nf.equipment == 'patient lift'].copy()
ASPECT = {
    'moving the device (pushing, steering, device rolling or tipping)':
        r"\b(?:push\w*|pull\w*|mov\w*|steer\w*|maneuver\w*) (?:the |a )?(?:\w+ )?lift\b|\blift (?:rolled|tipped|tilted|fell|bumped|hit|ran over)|ran over\b|rolled over (?:her|his|my|the employee|ee)|\btipped\b",
    'floor, threshold, incline, or space':
        r"\b(?:threshold|carpet|incline|ramp|slope|doorway|door frame|tight space|small (?:room|space|bathroom)|cramped|corner)\b",
    'power, battery, or malfunction':
        r"\b(?:battery|batteries|charg\w*|malfunction\w*|broke\w*|failed|not work\w*|stopped working|stuck|jammed|emergency release|power)\b",
    'sling or pad (placing, removing, positioning)':
        r"\b(?:sling|pad)\b",
    'resident behavior during lift use':
        r"\b(?:combative|aggressive|agitated|kick\w*|hit|punch\w*|bit|bite|grab\w*|scratch\w*|resist\w*|pull\w* away)\b",
    'catching or holding a resident during lift use':
        r"\b(?:fall\w*|fell|slid\w*|slipp\w*|catch\w*|caught)\b",
}
asp = pd.Series({k: int(L.narr.str.contains(p, regex=True).sum()) for k, p in ASPECT.items()}).rename('cases').to_frame()
asp['share of lift cases'] = (asp.cases / len(L)).round(3)
print(f'{len(L):,} cases mention a patient lift')
asp

724 cases mention a patient lift


,cases,share of lift cases
"moving the device (pushing, steering, device rolling or tipping)",121,0.167
"floor, threshold, incline, or space",12,0.017
"power, battery, or malfunction",62,0.086
"sling or pad (placing, removing, positioning)",137,0.189
resident behavior during lift use,104,0.144
catching or holding a resident during lift use,162,0.224


## 5. Severity by equipment group (handling cases)

In [6]:
h = nf[nf.handling]
sev = h.groupby('equipment').agg(cases=('narr', 'size'), days_away_sum=('dafw', 'sum'),
                                  median_days_away=('dafw', 'median'), over_90_days=('dafw', lambda s: int((s > 90).sum())))
sev

,cases,days_away_sum,median_days_away,over_90_days
equipment,,,,
low-tech aid,212,1812,0.0,6
none mentioned,4809,48547,0.0,175
patient lift,615,7144,0.0,30
vehicle lift,18,675,1.0,4


## 6. Validation sample

The equipment patterns are keyword rules and need the same check as the handling rule. The cell writes a blind sample: 100 cases flagged as patient lift and 100 random handling cases with no equipment mentioned, shuffled together, narrative only. A reviewer marks each as `lift used`, `aid used`, or `no equipment stated`. The result estimates how often the patterns are right, and how often equipment was involved but not caught.

In [7]:
s1 = nf[nf.equipment == 'patient lift'].sample(min(100, (nf.equipment == 'patient lift').sum()), random_state=2026)
s2 = nf[nf.handling & (nf.equipment == 'none mentioned')].sample(100, random_state=2026)
val = pd.concat([s1, s2]).sample(frac=1, random_state=7)[['narr']].assign(label='').reset_index(names='row')
val.to_csv('derived/equipment-validation-sample.csv', index=False)
print(len(val), 'cases written to derived/equipment-validation-sample.csv')

200 cases written to derived/equipment-validation-sample.csv


## 7. Export

In [8]:
tab.to_csv('derived/equipment-by-group.csv'); asp.to_csv('derived/lift-aspects.csv'); sev.to_csv('derived/severity-by-equipment.csv')
with open('derived/classification-rule.txt', 'w') as f:
    f.write(f'EQ_LIFT = {EQ_LIFT}\nEQ_AID  = {EQ_AID}\nEQ_VEH  = {EQ_VEH}\n\n'
            "vehicle lift = EQ_VEH\npatient lift = EQ_LIFT and not vehicle lift\nlow-tech aid = EQ_AID and not patient lift\n")
print(sorted(os.listdir('derived')))

['classification-rule.txt', 'equipment-by-group.csv', 'equipment-validation-sample.csv', 'lift-aspects.csv', 'severity-by-equipment.csv']
